In [2]:
import os, sys
from datetime import datetime
import uuid

from typing import TypedDict, Dict, Any, List, Literal, Optional, Annotated
from pydantic import Field, BaseModel
import operator

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  # or adjust path as needed

from src.logging_exception.logging.logger import logger
from src.logging_exception.exception.custom_exception import CustomException

from dotenv import load_dotenv
import json
import re

In [2]:
# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["SERPAPI_API_KEY"] = os.getenv("SERPAPI_API_KEY")

In [ ]:
# LLM client (kept as originally configured)

# llm = ChatGroq(model="openai/gpt-oss-120b", temperature=1)
#llm = ChatGroq(model="llama-3.1-8b-instant", temperature=1)
#llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=1)
llm = ChatGroq(model="moonshotai/kimi-k2-instruct-0905", temperature=1)    # works good for report generation
# llm = ChatGroq(model="meta-llama/llama-4-maverick-17b-128e-instruct", temperature=1)
# llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=1)

# GRAPH 1  - ANALYST CREATION

In [ ]:
class Analyst(BaseModel):
    role: str = Field(description="Role in the context of the restaurant_concept")
    affiliation: str = Field(description="Organisation the analyst belongs to")
    tools: List[str] = Field(description="Tools this analyst can use")
    description: str = Field(description="Responsibilities and focus areas")

class ResearchState(TypedDict, total=False):
    # ---------- USER INPUT -----------#
    restaurant_concept: str        # Used in: Graph1 (analyst generation), Graph2 (research queries)
    target_location: str           # Used in: Graph1 (analyst generation), Graph2 (location-specific research)
    investment: str                # Used in: Graph1 (analyst generation), Graph2 (financial analysis)
    target_demographic: str        # Used in: Graph1 (analyst generation), Graph2 (demographic research)
    special_note: str              # Used in: Graph1 (custom analyst generation), Graph2 (special research focus)

    # ---------- ANALYST SETUP -----------#
    max_analysts: int              # Used in: Graph1 (limit dynamic analyst count)
    fixed_analysts: List[Analyst]  # Used in: Graph1 (pre-defined analysts), Graph2 (research team)
    dynamic_analysts: List[Analyst] # Used in: Graph1 (generated analysts), Graph2 (research team)
    analysts_team: List[Analyst]   # Used in: Graph1 (final team), Graph2 (research execution)

    # ---------- HUMAN FEEDBACK -----------#
    accept: bool                   # Used in: Graph1 (team approval loop), Graph2 (research approval)
    extra_instructions: str        # Used in: Graph1 (regeneration guidance), Graph2 (human guidance)

In [ ]:
DYNAMIC_ANALYSTS_PROMPT = ChatPromptTemplate.from_template("""\
Generate EXACTLY {max_analysts} new analysts for a restaurant research team. 
They must add capabilities not present in the fixed analysts.

BUSINESS CONTEXT:
- restaurant_concept: {restaurant_concept}
- target_location: {target_location}
- Investment: {investment}
- target_demographic: {target_demographic}
- special_note: {special_note}

EXISTING FIXED ANALYSTS:
{existing_analysts}

RULES:
- Do not repeat or rename any fixed-analyst roles.
- Cover capability gaps only.
- Stay specific to the concept and location.
- Each analyst must be practically useful for market-research workflows.

Each analyst must include:
- "role": unique specialization name
- "affiliation": credible organisation
- "tools": 1-2 items from [market_competitor_analysis, location_demographic_analysis, financial_operations_analysis]
- "description": clear responsibility scope

IMPORTANT: Return ONLY a valid JSON array. No markdown formatting, no additional text, no explanations.
""")

In [ ]:
def load_fixed_analysts(state: ResearchState) -> Dict[str, List[Analyst]]:
    fixed_analysts = [
        Analyst(
            role="Market Analyst",
            affiliation="Market Intelligence Division",
            tools=["market_competitor_analysis"],
            description=(
                "Study demand signals, track industry trends, map opportunity gaps, "
                "evaluate seasonality patterns, quantify market size, assess product–market fit, "
                "validate demand assumptions and monitor macro shifts affecting category growth"
            )
        ),
        Analyst(
            role="Demographics Analyst",
            affiliation="Geographic Research Institute",
            tools=["location_demographic_analysis"],
            description=(
                "Analyze population distribution, map catchment areas, assess footfall density, "
                "evaluate urban–rural split, study socioeconomic clusters, examine household profiles, "
                "identify mobility patterns and validate locality suitability for target customer groups"
            )
        ),
        Analyst(
            role="Financial Analyst",
            affiliation="Financial Planning Department",
            tools=["financial_operations_analysis"],
            description=(
                "Build cost models, compute breakeven points, estimate ROI, project cash flows, "
                "evaluate payback periods, benchmark cost structures, run scenario simulations, "
                "assess pricing feasibility and validate financial viability"
            )
        ),
        Analyst(
            role="Competitor Analyst",
            affiliation="Competitive Intelligence Unit",
            tools=["market_competitor_analysis"],
            description=(
                "Identify direct and indirect competitors, analyze menus and pricing, study reviews, "
                "map competitor density, benchmark offerings, detect positioning gaps, "
                "track operational strengths/weaknesses and evaluate competitive pressure"
            )
        )
    ]
    return {"fixed_analysts": fixed_analysts}



def _extract_json_array(text: str) -> Optional[str]:
    """
    Robustly extract the first JSON array (top-level) from text.
    Returns the JSON string or None.
    """
    # Try a direct array search first
    array_match = re.search(r'(\[\s*\{.*?\}\s*\])', text, re.DOTALL)
    if array_match:
        return array_match.group(1)

    # Next, try to find inside a code block
    code_block_match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
    if code_block_match:
        inner = code_block_match.group(1).strip()
        arr_match = re.search(r'(\[\s*\{.*?\}\s*\])', inner, re.DOTALL)
        if arr_match:
            return arr_match.group(1)

    return None


def load_dynamic_analysts(state: ResearchState) -> Dict[str, List[Analyst]]:
    """
    LangGraph-compatible node.
    Reads state, calls prompt, returns {"dynamic_analysts": [...] }.
    """
    extra_note = state.get("extra_instructions", "") or ""

    existing_analysts_text = ""
    for a in state["fixed_analysts"]:
        existing_analysts_text += f"- {a.role}; {a.affiliation}; {a.tools}; {a.description}\n"

    prompt_text = DYNAMIC_ANALYSTS_PROMPT.format(
        max_analysts=state["max_analysts"],
        restaurant_concept=state["restaurant_concept"],
        target_location=state["target_location"],
        investment=state["investment"],
        target_demographic=state["target_demographic"],
        special_note=(state.get("special_note", "") + " " + extra_note).strip(),
        existing_analysts=existing_analysts_text,
    )

    # Invoke LLM
    result = llm.invoke(prompt_text)
    raw = getattr(result, "content", "") or ""

    # Try robust JSON extraction/parsing
    parsed_items = []
    json_candidate = _extract_json_array(raw)
    if json_candidate:
        try:
            parsed_items = json.loads(json_candidate)
        except json.JSONDecodeError:
            # Attempt to sanitize common LLM artifacts
            cleaned = json_candidate.replace("'", '"')
            try:
                parsed_items = json.loads(cleaned)
            except json.JSONDecodeError:
                logger.warning("Failed to parse JSON from LLM output.")
                parsed_items = []
    else:
        # final naive fallback: try parse entire content
        try:
            parsed_items = json.loads(raw)
        except Exception:
            parsed_items = []

    dynamic_analysts: List[Analyst] = []
    for item in parsed_items:
        if not isinstance(item, dict):
            continue
        try:
            dynamic_analysts.append(Analyst(**item))
        except Exception:
            logger.debug("Skipping invalid analyst item: %s", item)
            continue

    return {"dynamic_analysts": dynamic_analysts}


def human_feedback(state: ResearchState) -> Dict[str, Any]:

    print("\nFixed Analysts:")
    fixed_list = state.get("fixed_analysts", [])
    if fixed_list:
        for idx, analyst in enumerate(fixed_list, 1):
            print(f"\n{idx}. {analyst.role}")
            print(f"   Affiliation: {analyst.affiliation}")
            print(f"   Tools: {', '.join(analyst.tools)}")
            print(f"   Description: {analyst.description}")
    else:
        print("No fixed analysts found.")

    print("\nGenerated Dynamic Analysts:")
    dynamics = state.get("dynamic_analysts", [])
    if dynamics:
        for idx, analyst in enumerate(dynamics, 1):
            print(f"\n{idx}. {analyst.role}")
            print(f"   Affiliation: {analyst.affiliation}")
            print(f"   Tools: {', '.join(analyst.tools)}")
            print(f"   Description: {analyst.description}")
    else:
        print("No dynamic analysts generated!")

    try:
        user_input = input("\nAccept these Analysts? Enter 'y' for Yes or 'n' for No:  ").strip().lower()
    except Exception:
        user_input = "n"

    if user_input == 'y':
        return {"accept": True}

    try:
        extra_instructions = input("Give Extra Instructions (or press Enter to skip): ").strip()
    except Exception:
        extra_instructions = ""

    return {"accept": False, "extra_instructions": extra_instructions}



def combine_analysts(state: ResearchState) -> Dict[str, List[Analyst]]:
    """Combine fixed and dynamic analysts into final team"""
    all_analysts = list(state.get("fixed_analysts", [])) + list(state.get("dynamic_analysts", []))
    return {"analysts_team": all_analysts}


In [ ]:
# Build graph
graph = StateGraph(ResearchState)
graph.add_node("load_fixed_analysts", load_fixed_analysts)
graph.add_node("load_dynamic_analysts", load_dynamic_analysts)
graph.add_node("human_feedback", human_feedback)
graph.add_node("combine_analysts", combine_analysts)

graph.add_edge(START, "load_fixed_analysts")
graph.add_edge("load_fixed_analysts", "load_dynamic_analysts")
graph.add_edge("load_dynamic_analysts", "human_feedback")
#-------------------------------------------------------------
def route_feedback(state: ResearchState) -> str:
    return "combine" if state.get("accept") else "retry"
#-------------------------------------------------------------

graph.add_conditional_edges(
    "human_feedback",
    route_feedback,
    {"combine": "combine_analysts", "retry": "load_dynamic_analysts"},
)

graph.add_edge("combine_analysts", END)

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# Initial state - example
initial_state: ResearchState = {
    "restaurant_concept": "Setup momo speciality Restaurant",
    "target_location": "Sneha Paradise-A, Warje-Malwadi, Pune, Maharashtra - 411058",
    "investment": "₹350,000",
    "target_demographic": "Professionals 25-45, health-conscious",
    "special_note": "momo demand, spending power, competition density",
    "max_analysts": 2
}

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

final_state = app.invoke(initial_state, config=config)

print("\n" + "="*50)
print("FINAL RESEARCH TEAM")
print("="*50)
for idx, analyst in enumerate(final_state.get("analysts_team", []), 1):
    print(f"\n{idx}. {analyst.role}")
    print(f"   Affiliation: {analyst.affiliation}")
    print(f"   Tools: {', '.join(analyst.tools)}")
    print(f"   Description: {analyst.description}")


# TOOL SETUP

In [1]:
from langchain_community.tools import TavilySearchResults
import requests
from typing import Dict

# Initialize tools
tavily = TavilySearchResults(max_results=5)

def use_tavily_search(state: Dict) -> Dict:
    """Universal web search for any information"""
    query = state.get("search_query", "")
    if not query:
        return {"error": "no search query"}
    try:
        results = tavily.invoke({"query": query})
        return {
            "tool": "tavily_search",
            "results": results,
            "sources_count": len(results)
        }
    except Exception as e:
        return {"error": f"tavily failed: {str(e)}"}

def use_google_trends(state: Dict) -> Dict:
    """Trend analysis with Tavily fallback"""
    query = state.get("trend_query", "")
    if not query:
        return {"error": "no trend query"}
    return use_tavily_search({
        "search_query": f"trends popularity analysis {query} 2024"
    })

def use_openstreetmap(state: Dict) -> Dict:
    """Location data and coordinates"""
    location = state.get("location", "").strip()
    if not location:
        return {"error": "no location"}
    try:
        response = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={"q": location, "format": "json", "limit": 1, "countrycodes": "in"},
            headers={"User-Agent": "ResearchApp/1.0"},
            timeout=10
        )
        if response.status_code == 200 and response.json():
            place = response.json()[0]
            return {
                "tool": "openstreetmap",
                "location": location,
                "latitude": place.get("lat"),
                "longitude": place.get("lon"),
                "display_name": place.get("display_name"),
                "sources_count": 1
            }
        return use_tavily_search({"search_query": f"location information {location}"})
    except Exception:
        return use_tavily_search({"search_query": f"location {location}"})

def use_google_maps(state: Dict) -> Dict:
    """Nearby places and businesses"""
    location = state.get("location", "")
    query = state.get("query", "")
    search_term = location or query
    return use_tavily_search({
        "search_query": f"{search_term} nearby places restaurants businesses"
    })

def select_and_use_tool(state: Dict) -> Dict:
    """Smart tool selection based on task type"""
    task = state.get("task_type", "").lower()
    query = state.get("query", "").lower().strip()
    
    if not query:
        return use_tavily_search({"search_query": "current trends"})
    
    # Tool selection logic
    if any(word in task or word in query for word in ["trend", "popular", "viral"]):
        result = use_google_trends({"trend_query": query})
        result["tool_selected"] = "google_trends"
    elif any(word in task or word in query for word in ["location", "map", "where is", "address"]):
        result = use_openstreetmap({"location": query})
        result["tool_selected"] = "openstreetmap"
    elif any(word in task or word in query for word in ["nearby", "restaurant", "cafe", "shop"]):
        result = use_google_maps({"location": query})
        result["tool_selected"] = "google_maps"
    else:
        result = use_tavily_search({"search_query": query})
        result["tool_selected"] = "tavily_search"
    
    return result

# Test the functions
if __name__ == "__main__":
    print("Testing tools...")
    
    # Test 1: Tavily Search
    print("\n1. Testing Tavily Search:")
    result1 = use_tavily_search({"search_query": "Pune restaurants"})
    print(f"Status: {'Success' if 'error' not in result1 else 'Failed'}")
    print(f"Sources: {result1.get('sources_count', 0)}")
    
    # Test 2: Google Trends (with fallback)
    print("\n2. Testing Google Trends:")
    result2 = use_google_trends({"trend_query": "biryani"})
    print(f"Tool used: {result2.get('tool', 'unknown')}")
    print(f"Sources: {result2.get('sources_count', 0)}")
    
    # Test 3: OpenStreetMap
    print("\n3. Testing OpenStreetMap:")
    result3 = use_openstreetmap({"location": "Bangalore Karnataka"})
    print(f"Tool: {result3.get('tool', 'unknown')}")
    if 'display_name' in result3:
        print(f"Found: {result3['display_name']}")
    else:
        print(f"Fallback to: {result3.get('tool', 'search')}")
    
    # Test 4: Auto-selection
    print("\n4. Testing Auto-selection:")
    test_cases = [
        {"task_type": "find news", "query": "restaurant trends"},
        {"task_type": "check trends", "query": "popular foods"},
        {"task_type": "get location", "query": "Mumbai"},
        {"task_type": "find nearby", "query": "coffee shops"}
    ]
    
    for i, test in enumerate(test_cases, 1):
        result = select_and_use_tool(test)
        tool = result.get('tool_selected', 'tavily_search')
        sources = result.get('sources_count', 0)
        print(f"  Case {i}: '{test['query']}' -> {tool} ({sources} sources)")

C:\Users\gawal\AppData\Local\Temp\ipykernel_13912\4017448531.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(max_results=5)


Testing tools...

1. Testing Tavily Search:
Status: Success
Sources: 5

2. Testing Google Trends:
Tool used: tavily_search
Sources: 5

3. Testing OpenStreetMap:
Tool: openstreetmap
Found: Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, India

4. Testing Auto-selection:
  Case 1: 'restaurant trends' -> google_trends (5 sources)
  Case 2: 'popular foods' -> google_trends (5 sources)
  Case 3: 'Mumbai' -> openstreetmap (1 sources)
  Case 4: 'coffee shops' -> google_maps (5 sources)


# GRAPH 2 - PARALLEL DATA COLLECTION

In [ ]:
from typing import TypedDict, List, Dict, Any, Optional, Annotated
from pydantic import BaseModel, Field
import operator

class Analyst(BaseModel):
    role: str = Field(description="Role in the context of the restaurant_concept")
    affiliation: str = Field(description="Organisation the analyst belongs to")
    tools: List[str] = Field(description="Tools this analyst can use")
    description: str = Field(description="Responsibilities and focus areas")

class ResearchState(TypedDict, total=False):
    # ---------- USER INPUT -----------#
    restaurant_concept: str        # Used in: Graph1 (analyst generation), Graph2 (research queries)
    target_location: str           # Used in: Graph1 (analyst generation), Graph2 (location-specific research)
    investment: str                # Used in: Graph1 (analyst generation), Graph2 (financial analysis)
    target_demographic: str        # Used in: Graph1 (analyst generation), Graph2 (demographic research)
    special_note: str              # Used in: Graph1 (custom analyst generation), Graph2 (special research focus)

    # ---------- ANALYST SETUP -----------#
    max_analysts: int              # Used in: Graph1 (limit dynamic analyst count)
    fixed_analysts: List[Analyst]  # Used in: Graph1 (pre-defined analysts), Graph2 (research team)
    dynamic_analysts: List[Analyst] # Used in: Graph1 (generated analysts), Graph2 (research team)
    analysts_team: List[Analyst]   # Used in: Graph1 (final team), Graph2 (research execution)

    # ---------- HUMAN FEEDBACK -----------#
    accept: bool                   # Used in: Graph1 (team approval loop), Graph2 (research approval)
    extra_instructions: str        # Used in: Graph1 (regeneration guidance), Graph2 (human guidance)

    # ---------- RESEARCH EXECUTION -----------#
    current_analyst: Optional[Analyst]          # Used in: Graph2 (track current analyst in iteration)
    research_queries: Annotated[Dict[str, List[str]], operator.add]  # Used in: Graph2 (store queries per analyst)
    research_results: Annotated[Dict[str, List[Dict]], operator.add] # Used in: Graph2 (store results per analyst)
    tool_executions: Annotated[List[Dict], operator.add]             # Used in: Graph2 (audit trail of tool usage)
    
    # ---------- PROGRESS & CONTROL -----------#
    research_complete: bool        # Used in: Graph2 (termination condition)
    human_feedback_required: bool  # Used in: Graph2 (conditional routing for human review)
    human_instructions: str        # Used in: Graph2 (carry human input to next nodes)
    iteration_count: int           # Used in: Graph2 (track analyst progress, conditional routing)
    errors: Annotated[List[str], operator.add]  # Used in: Graph2 (collect errors without breaking flow)
    
    # ---------- FINAL OUTPUT -----------#
    executive_summary: str         # Used in: Graph2 (final synthesis output)
    recommendations: List[str]     # Used in: Graph2 (actionable recommendations output)
    risk_assessment: Dict[str, Any] # Used in: Graph2 (structured risk analysis output)